# WNBA Draft Fit Predictor — Feature Engineering

This notebook builds the features needed for modeling:
1. Rookie Impact Score (RIS) — our target variable
2. Prospect career features — weighted NCAA career averages
3. Team context features — what each team needs

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
rookies_df = pd.read_csv("../data/processed/rookies_clean.csv")
prospects_df = pd.read_csv("../data/processed/prospects_2026_clean.csv")
teams_df = pd.read_csv("../data/processed/team_stats_clean.csv")

print("Rookies:", rookies_df.shape)
print("Prospects:", prospects_df.shape)
print("Teams:", teams_df.shape)

Rookies: (96, 13)
Prospects: (51, 31)
Teams: (85, 25)


In [3]:
# Build Rookie Impact Score (RIS)
# Weighted combination of: games played, minutes, scoring, efficiency

# Normalize each component to 0-1 scale first
from sklearn.preprocessing import MinMaxScaler

ris_features = ['wnba_games', 'wnba_mpg', 'wnba_ppg', 'wnba_rpg', 'wnba_apg', 'wnba_ws40']

scaler = MinMaxScaler()
rookies_scaled = rookies_df.copy()
rookies_scaled[ris_features] = scaler.fit_transform(rookies_df[ris_features])

# Weighted RIS — availability + usage + production + efficiency
rookies_df['RIS'] = (
    0.20 * rookies_scaled['wnba_games'] +    # availability
    0.20 * rookies_scaled['wnba_mpg'] +      # usage/trust from coach
    0.20 * rookies_scaled['wnba_ppg'] +      # scoring
    0.15 * rookies_scaled['wnba_rpg'] +      # rebounding
    0.10 * rookies_scaled['wnba_apg'] +      # playmaking
    0.15 * rookies_scaled['wnba_ws40']       # efficiency
).round(3)

print("RIS Summary:")
print(rookies_df['RIS'].describe().round(3))
print("\nTop 10 rookies by RIS:")
print(rookies_df[['player', 'draft_year', 'wnba_team', 'RIS']].sort_values('RIS', ascending=False).head(10))

RIS Summary:
count    96.000
mean      0.363
std       0.186
min       0.064
25%       0.227
50%       0.337
75%       0.476
max       0.809
Name: RIS, dtype: float64

Top 10 rookies by RIS:
              player  draft_year         wnba_team    RIS
5   Napheesa Collier        2019    Minnesota Lynx  0.809
4   Arike Ogunbowale        2019      Dallas Wings  0.786
15   Sabrina Ionescu        2020  New York Liberty  0.761
0       Jackie Young        2019    Las Vegas Aces  0.745
71     Caitlin Clark        2024     Indiana Fever  0.705
44      Rhyne Howard        2022     Atlanta Dream  0.696
58     Aliyah Boston        2023     Indiana Fever  0.696
16     Satou Sabally        2020      Dallas Wings  0.680
77       Angel Reese        2024       Chicago Sky  0.676
84    Paige Bueckers        2025      Dallas Wings  0.653


In [5]:
def weighted_career_stats(group):
    n = len(group)
    weights = np.arange(1, n + 1)
    weights = weights / weights.sum()
    
    stats = ['G', 'MP', 'FG%', '3P%', 'FT%', 'ORB', 'DRB', 'TRB', 
             'AST', 'STL', 'BLK', 'TOV', 'PTS', 'eFG%']
    
    result = {}
    for stat in stats:
        result[f'avg_{stat}'] = np.average(group[stat], weights=weights)
    
    result['seasons_played'] = n
    result['final_season_pts'] = group['PTS'].iloc[-1]
    
    return pd.Series(result)

prospects_features = prospects_df.sort_values(['player', 'Season']).groupby('player').apply(weighted_career_stats).reset_index()

print(f"Prospect features shape: {prospects_features.shape}")
print(prospects_features[['player', 'avg_PTS', 'avg_TRB', 'avg_AST', 'avg_FG%']].round(2))

Prospect features shape: (12, 17)
              player  avg_PTS  avg_TRB  avg_AST  avg_FG%
0     Angela Dugalic     7.81     5.50     1.93     0.44
1          Azzi Fudd    14.41     2.33     2.33     0.44
2      Cotie McMahon    17.14     5.26     2.49     0.46
3   Flau'jae Johnson    15.34     5.05     2.44     0.47
4    Gabriela Jaquez    10.91     5.26     1.88     0.51
5   Gianna Kneepkens    15.80     4.45     2.91     0.52
6          Kiki Rice    13.60     5.00     4.42     0.47
7       Lauren Betts    16.47     8.58     2.34     0.62
8        Madina Okot    12.30    10.27     0.90     0.60
9       Olivia Miles    15.95     6.36     6.35     0.48
10     Raven Johnson     6.79     3.95     3.86     0.39
11        Taina Mair     9.72     4.49     4.70     0.40


In [6]:
# Save prospect features
prospects_features.to_csv("../data/processed/prospect_features.csv", index=False)
print("Saved prospect_features.csv!")

# Now build team context features for 2025 (the team context going into 2026)
teams_2025 = teams_df[teams_df['season'] == 2025].copy()

# Map 2026 draft teams to their team stats
draft_teams = {
    "Azzi Fudd": "Dallas Wings",
    "Olivia Miles": "Minnesota Lynx",
    "Lauren Betts": "Washington Mystics",
    "Gabriela Jaquez": "Chicago Sky",
    "Kiki Rice": "Toronto Tempo",
    "Flau'jae Johnson": "Seattle Storm",
    "Angela Dugalic": "Washington Mystics",
    "Raven Johnson": "Indiana Fever",
    "Cotie McMahon": "Washington Mystics",
    "Madina Okot": "Atlanta Dream",
    "Taina Mair": "Seattle Storm",
    "Gianna Kneepkens": "Connecticut Sun",
}

print("Draft team mapping created!")
print(teams_2025[['Team', 'PTS', 'TRB', 'AST']].head())

Saved prospect_features.csv!
Draft team mapping created!
                  Team   PTS   TRB   AST
72      Minnesota Lynx  86.1  34.2  23.3
73  Los Angeles Sparks  85.7  33.3  20.7
74       Indiana Fever  84.9  33.4  20.6
75       Atlanta Dream  84.4  36.6  21.4
76    New York Liberty  84.4  33.7  21.8
